# 01 — Run Frame Selection Experiment

**Purpose.** Plug in a frame selector, run it against the cached LongVideoBench subset with Qwen2-VL-2B as the judge, and get accuracy numbers.

**Prerequisites (Kaggle setup):**
1. Attach the **lvb-cache** dataset (built by notebook `00_build_lvb_cache.ipynb`).
2. Attach the code as a Kaggle Dataset (`kaggle-frame-eval`) OR clone from your GitHub repo in the first cell.
3. Enable **GPU T4 x2** (or P100).
4. Add `HF_TOKEN` as a Kaggle Secret if you want to download models faster from gated repos.

## 1. Set up paths and dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes pyarrow

In [ ]:
import sys, os
from pathlib import Path

# ===== EDIT THESE TWO PATHS =====
CODE_DIR  = "/kaggle/input/kaggle-frame-eval"   # the dataset containing this repo's .py files
CACHE_DIR = "/kaggle/input/lvb-cache/lvb-cache" # path to the cache built by notebook 00
# =================================

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Sanity check
assert Path(CODE_DIR).exists(), f"Code dir not found: {CODE_DIR}"
assert Path(CACHE_DIR).exists(), f"Cache dir not found: {CACHE_DIR}"
print("Cache contents:", list(Path(CACHE_DIR).iterdir())[:5])

## 2. Configure the run

Three things define a run: **selector**, **k** (frames the VLM sees), and **n_examples** (subset size). The defaults below are the smoke configuration.

In [ ]:
# ===== RUN CONFIG =====
SELECTOR_NAME = "uniform"      # one of: 'uniform', 'random', 'siglip_topk'
K_FRAMES      = 8              # number of frames the VLM sees (paper standard: 8)
N_EXAMPLES    = 10             # smoke: 10. dev: 100. full: None.
RUN_NAME      = f"{SELECTOR_NAME}_k{K_FRAMES}_n{N_EXAMPLES}"
OUTPUT_DIR    = f"/kaggle/working/results"
VLM_MODEL_ID  = "Qwen/Qwen2-VL-2B-Instruct"
SEED          = 0
# ======================

## 3. Load dataset, selector, and VLM

In [ ]:
from data import LVBCache
from frame_selectors import build_selector
from vlm import Qwen2VLJudge
from eval import run_evaluation

# Dataset
dataset = LVBCache(CACHE_DIR).subset(n=N_EXAMPLES, seed=SEED)
print(f"Dataset: {len(dataset)} examples")

In [ ]:
# Selector
if SELECTOR_NAME == "random":
    selector = build_selector(SELECTOR_NAME, seed=SEED)
else:
    selector = build_selector(SELECTOR_NAME)
print("Selector:", selector)

In [ ]:
# VLM (this load is the slowest step — caches in /root/.cache after first run)
judge = Qwen2VLJudge(model_id=VLM_MODEL_ID)
print("VLM loaded:", VLM_MODEL_ID)

## 4. Run the evaluation

In [ ]:
metrics = run_evaluation(
    dataset=dataset,
    selector=selector,
    judge=judge,
    k=K_FRAMES,
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
)

## 5. Inspect per-example results

Useful for debugging: look at cases where the selector and the VLM disagreed with the ground truth.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/per_example_{RUN_NAME}.csv")
df['correct'] = (df['pred_idx'] == df['correct_idx']).astype(int)

print("Wrong answers (sample of 5):")
df[df['correct'] == 0].head()

## 6. Compare two selectors side-by-side

This is the sanity test: **uniform should beat random** on enough samples to be statistically meaningful. If it doesn't, scale up `N_EXAMPLES` (random sampling has high variance on small n).

In [ ]:
# Re-run with the random selector for comparison.
# (We keep the same judge — loading it once and reusing saves GPU time.)
random_selector = build_selector("random", seed=SEED)
metrics_random = run_evaluation(
    dataset=dataset,
    selector=random_selector,
    judge=judge,
    k=K_FRAMES,
    output_dir=OUTPUT_DIR,
    run_name=f"random_k{K_FRAMES}_n{N_EXAMPLES}",
)

In [ ]:
print("=== Side-by-side ===")
print(f"{SELECTOR_NAME:>15s}: {metrics['overall']['accuracy']:.4f}")
print(f"{'random':>15s}: {metrics_random['overall']['accuracy']:.4f}")

## When you develop your own algorithm

1. Add a file `selectors/my_algo.py` with a class that subclasses `FrameSelector` and implements `select(frames, query, k) -> List[int]`.
2. Register it in `selectors/__init__.py` by adding it to the `SELECTORS` dict.
3. Upload the updated code as a new version of the `kaggle-frame-eval` dataset.
4. Set `SELECTOR_NAME = "my_algo"` in the config cell above and rerun.

Nothing else changes. Same dataset cache, same VLM, same metrics.